# Load and parse data with TensorFlow 2.0 (tf.data)

A TensorFlow 2.0 example to build input pipelines for loading data efficiently.


- Numpy Arrays
- Images
- CSV file
- Custom data from a Generator

For more information about creating and loading TensorFlow's `TFRecords` data format, see: [tfrecords.ipynb](tfrecords.ipynb)

In [1]:
from __future__ import absolute_import, division, print_function

import numpy as np
import random
import requests
import string
import tarfile
import tensorflow as tf

### Load Numpy Arrays

Build a data pipeline over numpy arrays.

In [2]:
# Create a toy dataset (even and odd numbers, with respective labels of 0 and 1).
evens = np.arange(0, 100, step=2, dtype=np.int32)
evens_label = np.zeros(50, dtype=np.int32)
odds = np.arange(1, 100, step=2, dtype=np.int32)
odds_label = np.ones(50, dtype=np.int32)
# Concatenate arrays
features = np.concatenate([evens, odds])
labels = np.concatenate([evens_label, odds_label])

# Load a numpy array using tf data api with `from_tensor_slices`.
data = tf.data.Dataset.from_tensor_slices((features, labels))
# Refill data indefinitely.  
data = data.repeat()
# Shuffle data.
data = data.shuffle(buffer_size=100)
# Batch data (aggregate records together).
data = data.batch(batch_size=4)
# Prefetch batch (pre-load batch for faster consumption).
data = data.prefetch(buffer_size=1)

In [3]:
for batch_x, batch_y in data.take(5):
    print(batch_x, batch_y)

tf.Tensor([99 15  2 10], shape=(4,), dtype=int32) tf.Tensor([1 1 0 0], shape=(4,), dtype=int32)
tf.Tensor([30 21 60 28], shape=(4,), dtype=int32) tf.Tensor([0 1 0 0], shape=(4,), dtype=int32)
tf.Tensor([63 97 16 68], shape=(4,), dtype=int32) tf.Tensor([1 1 0 0], shape=(4,), dtype=int32)
tf.Tensor([14 61 19 56], shape=(4,), dtype=int32) tf.Tensor([0 1 1 0], shape=(4,), dtype=int32)
tf.Tensor([44  4 24 64], shape=(4,), dtype=int32) tf.Tensor([0 0 0 0], shape=(4,), dtype=int32)


In [4]:
# Note: If you are planning on calling multiple time,
# you can user the iterator way:
ite_data = iter(data)
for i in range(5):
    batch_x, batch_y = next(ite_data)
    print(batch_x, batch_y)

for i in range(5):
    batch_x, batch_y = next(ite_data)
    print(batch_x, batch_y)

tf.Tensor([59 43 70 22], shape=(4,), dtype=int32) tf.Tensor([1 1 0 0], shape=(4,), dtype=int32)
tf.Tensor([18 65 36 54], shape=(4,), dtype=int32) tf.Tensor([0 1 0 0], shape=(4,), dtype=int32)
tf.Tensor([20 15 91 14], shape=(4,), dtype=int32) tf.Tensor([0 1 1 0], shape=(4,), dtype=int32)
tf.Tensor([ 0  6 56 18], shape=(4,), dtype=int32) tf.Tensor([0 0 0 0], shape=(4,), dtype=int32)
tf.Tensor([92 67 55  8], shape=(4,), dtype=int32) tf.Tensor([0 1 1 0], shape=(4,), dtype=int32)
tf.Tensor([53 19 84 24], shape=(4,), dtype=int32) tf.Tensor([1 1 0 0], shape=(4,), dtype=int32)
tf.Tensor([60 44 45 41], shape=(4,), dtype=int32) tf.Tensor([0 0 1 1], shape=(4,), dtype=int32)
tf.Tensor([11 33 44 48], shape=(4,), dtype=int32) tf.Tensor([1 1 0 0], shape=(4,), dtype=int32)
tf.Tensor([48 46 62 42], shape=(4,), dtype=int32) tf.Tensor([0 0 0 0], shape=(4,), dtype=int32)
tf.Tensor([68 73 49  4], shape=(4,), dtype=int32) tf.Tensor([0 1 1 0], shape=(4,), dtype=int32)


### Load CSV files

Build a data pipeline from features stored in a CSV file. For this example, Titanic dataset will be used as a toy dataset stored in CSV format.

#### Titanic Dataset



survived|pclass|name|sex|age|sibsp|parch|ticket|fare
--------|------|----|---|---|-----|-----|------|----
1|1|"Allen, Miss. Elisabeth Walton"|female|29|0|0|24160|211.3375
1|1|"Allison, Master. Hudson Trevor"|male|0.9167|1|2|113781|151.5500
0|1|"Allison, Miss. Helen Loraine"|female|2|1|2|113781|151.5500
0|1|"Allison, Mr. Hudson Joshua Creighton"|male|30|1|2|113781|151.5500
...|...|...|...|...|...|...|...|...

In [5]:
# Download Titanic dataset (in csv format).
d = requests.get("https://raw.githubusercontent.com/tflearn/tflearn.github.io/master/resources/titanic_dataset.csv")
with open("titanic_dataset.csv", "wb") as f:
    f.write(d.content)

In [6]:
# Load Titanic dataset.
# Original features: survived,pclass,name,sex,age,sibsp,parch,ticket,fare
# Select specific columns: survived,pclass,name,sex,age,fare
column_to_use = [0, 1, 2, 3, 4, 8]
record_defaults = [tf.int32, tf.int32, tf.string, tf.string, tf.float32, tf.float32]

# Load the whole dataset file, and slice each line.
data = tf.data.experimental.CsvDataset("titanic_dataset.csv", record_defaults, header=True, select_cols=column_to_use)
# Refill data indefinitely.
data = data.repeat()
# Shuffle data.
data = data.shuffle(buffer_size=1000)
# Batch data (aggregate records together).
data = data.batch(batch_size=2)
# Prefetch batch (pre-load batch for faster consumption).
data = data.prefetch(buffer_size=1)

In [7]:
for survived, pclass, name, sex, age, fare in data.take(1):
    print(survived.numpy())
    print(pclass.numpy())
    print(name.numpy())
    print(sex.numpy())
    print(age.numpy())
    print(fare.numpy())

[1 0]
[3 2]
[b'Andersen-Jensen, Miss. Carla Christine Nielsine'
 b'Frost, Mr. Anthony Wood "Archie"']
[b'female' b'male']
[19.  0.]
[7.8542 0.    ]


### Load Images

Build a data pipeline by loading images from disk. For this example, Oxford Flowers dataset will be used.

In [8]:
# Download Oxford 17 flowers dataset
d = requests.get("http://www.robots.ox.ac.uk/~vgg/data/flowers/17/17flowers.tgz")
with open("17flowers.tgz", "wb") as f:
    f.write(d.content)
# Extract archive.
with tarfile.open("17flowers.tgz") as t:
    t.extractall()

In [9]:
with open('jpg/dataset.csv', 'w') as f:
    c = 0
    for i in range(1360):
        f.write("jpg/image_%04i.jpg,%i\n" % (i+1, c))
        if (i+1) % 80 == 0:
            c += 1

In [10]:
# Load Images
with open("jpg/dataset.csv") as f:
    dataset_file = f.read().splitlines()

# Load the whole dataset file, and slice each line.
data = tf.data.Dataset.from_tensor_slices(dataset_file)
# Refill data indefinitely.
data = data.repeat()
# Shuffle data.
data = data.shuffle(buffer_size=1000)

# Load and pre-process images.
def load_image(path):
    # Read image from path.
    image = tf.io.read_file(path)
    # Decode the jpeg image to array [0, 255].
    image = tf.image.decode_jpeg(image)
    # Resize images to a common size of 256x256.
    image = tf.image.resize(image, [256, 256])
    # Rescale values to [-1, 1].
    image = 1. - image / 127.5
    return image
# Decode each line from the dataset file.
def parse_records(line):
    # File is in csv format: "image_path,label_id".
    # TensorFlow requires a default value, but it will never be used.
    image_path, image_label = tf.io.decode_csv(line, ["", 0])
    # Apply the function to load images.
    image = load_image(image_path)
    return image, image_label
# Use 'map' to apply the above functions in parallel.
data = data.map(parse_records, num_parallel_calls=4)

# Batch data (aggregate images-array together).
data = data.batch(batch_size=2)
# Prefetch batch (pre-load batch for faster consumption).
data = data.prefetch(buffer_size=1)

In [11]:
for batch_x, batch_y in data.take(1):
    print(batch_x, batch_y)

tf.Tensor(
[[[[ 7.2377455e-01  6.1397058e-01  8.1004900e-01]
   [ 7.2812307e-01  6.1831915e-01  8.1439757e-01]
   [ 7.3567903e-01  6.2587512e-01  8.2195354e-01]
   ...
   [ 9.0287989e-01  8.2444853e-01  8.8719362e-01]
   [ 9.1544116e-01  8.3700979e-01  8.9975488e-01]
   [ 9.1544116e-01  8.3700979e-01  8.9975488e-01]]

  [[ 7.4117649e-01  6.3137257e-01  8.3014703e-01]
   [ 7.4001801e-01  6.3021410e-01  8.2898855e-01]
   [ 7.3848039e-01  6.2867647e-01  8.2745099e-01]
   ...
   [ 9.0287989e-01  8.2444853e-01  8.8719362e-01]
   [ 9.1102940e-01  8.3259803e-01  8.9534312e-01]
   [ 9.1102940e-01  8.3259803e-01  8.9534312e-01]]

  [[ 7.2622550e-01  6.0931373e-01  8.3455884e-01]
   [ 7.2285539e-01  6.0594362e-01  8.3118874e-01]
   [ 7.1566141e-01  5.9874964e-01  8.2399470e-01]
   ...
   [ 9.0287989e-01  8.2444853e-01  8.8719362e-01]
   [ 9.0588236e-01  8.2745099e-01  8.9019608e-01]
   [ 9.0588236e-01  8.2745099e-01  8.9019608e-01]]

  ...

  [[ 5.5142653e-01  2.9967642e-01  9.5743144e-01]
   [ 

### Load data from a Generator

In [15]:
# Create a dummy generator.
def generate_features():
    # Function to generate a random string.
    def random_string(length):
        return ''.join(random.choice(string.ascii_letters) for m in range(length))
    # Return a random string, a random vector, and a random int.
    yield random_string(4), np.random.uniform(size=4), random.randint(0, 10)

In [16]:
# Load a numpy array using tf data api with `from_tensor_slices`.
data = tf.data.Dataset.from_generator(generate_features, output_types=(tf.string, tf.float32, tf.int32))
# Refill data indefinitely.
data = data.repeat()
# Shuffle data.
data = data.shuffle(buffer_size=100)
# Batch data (aggregate records together).
data = data.batch(batch_size=4)
# Prefetch batch (pre-load batch for faster consumption).
data = data.prefetch(buffer_size=1)

In [17]:
# Display data.
for batch_str, batch_vector, batch_int in data.take(5):
    print(batch_str, batch_vector, batch_int)

tf.Tensor([b'MqTH' b'ZsEu' b'QEQS' b'JQIm'], shape=(4,), dtype=string) tf.Tensor(
[[0.95711243 0.21215123 0.15214215 0.7933636 ]
 [0.83708614 0.8079769  0.72172123 0.27493197]
 [0.55606663 0.21029705 0.15011783 0.45774138]
 [0.32631367 0.4712611  0.7488375  0.4868973 ]], shape=(4, 4), dtype=float32) tf.Tensor([0 3 3 6], shape=(4,), dtype=int32)
tf.Tensor([b'JmuF' b'cBpS' b'uPYR' b'MenH'], shape=(4,), dtype=string) tf.Tensor(
[[0.3790001  0.24657364 0.11491779 0.44192016]
 [0.9855562  0.7846584  0.03978373 0.45925298]
 [0.02655213 0.9674962  0.36698943 0.2578532 ]
 [0.3852754  0.6793012  0.98352593 0.9023376 ]], shape=(4, 4), dtype=float32) tf.Tensor([7 7 7 2], shape=(4,), dtype=int32)
tf.Tensor([b'wPDK' b'tSjQ' b'BBDN' b'rVGb'], shape=(4,), dtype=string) tf.Tensor(
[[0.08791591 0.5575132  0.17732507 0.97349995]
 [0.24912935 0.4785967  0.18086317 0.00130618]
 [0.19085121 0.685073   0.5236383  0.01700619]
 [0.9503108  0.3752658  0.40666115 0.27322787]], shape=(4, 4), dtype=float32) tf.Te